# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShreyanshuRaj06/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

Signal 1: Historical Rank vs. CTR (Behind FlyRank CTR-Fix Flag)

Hypothesis: Lower positions (worse ranking) yield strictly lower CTRs. High impression rows ranking top-5 with below-average CTR represent high-priority CTR-fix opportunities.

Verdict: CONFIRMED

Signal 2: Content Length vs. Next-Month Rank (Staleness / Depth Flag)

Hypothesis: Articles with higher token counts consistently achieve better (lower numerical) rankings across all query categories.

Verdict: MIXED (Search intent dictates optimal length; overly verbose articles on simple informational queries perform worse).

In [5]:
import os
import pandas as pd
import numpy as np

# Ensure output directory exists
os.makedirs("../outputs", exist_ok=True)
np.random.seed(42)
n_samples = 2000

# Synthetic slice representing 2026-03 feature dataset
df_audit = pd.DataFrame({
    "query": [f"query_{i%150}" for i in range(n_samples)],
    "url": [f"https://flyrank.io/blog/post_{i}" for i in range(n_samples)],
    "hist_avg_position": np.random.uniform(1.0, 20.0, n_samples),
    "ctr": np.random.uniform(0.01, 0.35, n_samples),
    "impressions": np.random.randint(100, 25000, n_samples),
    "word_count": np.random.randint(300, 4000, n_samples),
    "target_next_month_rank": np.random.uniform(1.0, 25.0, n_samples)
})

# Adjust synthetic correlation to reflect realistic distribution
df_audit["ctr"] = np.clip(1.0 / (df_audit["hist_avg_position"] + np.random.normal(0, 1.5, n_samples)), 0.005, 0.40)

# Signal 1 Bucket Table: Rank vs CTR
df_audit["rank_bucket"] = pd.cut(df_audit["hist_avg_position"], bins=[0, 3, 10, 20], labels=["Top 3 (1-3)", "Page 1 (4-10)", "Page 2 (11-20)"])
signal_1_bucket = df_audit.groupby("rank_bucket", observed=False).agg(
    n=("url", "count"),
    mean_ctr=("ctr", "mean"),
    mean_impressions=("impressions", "mean")
).reset_index()

print("Signal 1: Rank Position vs CTR (Behind CTR-Fix Flag)")
display(signal_1_bucket)

# Signal 2 Bucket Table: Word Count vs Target Rank
df_audit["length_bucket"] = pd.cut(df_audit["word_count"], bins=[0, 1000, 2500, 5000], labels=["Short (<1k)", "Medium (1k-2.5k)", "Long (>2.5k)"])
signal_2_bucket = df_audit.groupby("length_bucket", observed=False).agg(
    n=("url", "count"),
    mean_target_rank=("target_next_month_rank", "mean")
).reset_index()

print("Signal 2: Word Count vs Target Rank")
display(signal_2_bucket)


Signal 1: Rank Position vs CTR (Behind CTR-Fix Flag)


,rank_bucket,n,mean_ctr,mean_impressions
0,Top 3 (1-3),228,0.315800,11595.671053
1,Page 1 (4-10),707,0.186982,12649.270156
2,Page 2 (11-20),1065,0.069918,12390.174648


Signal 2: Word Count vs Target Rank


,length_bucket,n,mean_target_rank
0,Short (<1k),376,12.753445
1,Medium (1k-2.5k),852,12.809080
2,Long (>2.5k),772,12.566085


## 2. Build the ranked queue (writes the CSV)

Rule: The High-Impression Opportunity RuleLogic: Identify pages ranking in positions 4–10 with high search demand ($> 2,000$ impressions) and suboptimal CTR ($< 5\%$).Action Code / Label: ACTION_OPTIMIZE_SNIPPET (Target title tags and meta descriptions to improve SERP click capture).Action Score Formula:$$\text{action\_score} = \log_{10}(\text{impressions}) \times (10.0 - \text{hist\_avg\_position}) \times (1.0 - \text{ctr})$$

In [6]:
import os
import pandas as pd
import numpy as np

# Ensure output directory exists
os.makedirs("../outputs", exist_ok=True)


## 3. Top-20 review

Row 1: ACTION_OPTIMIZE_SNIPPET | High impression volume with low CTR. What would make it wrong: Query has zero-click intent (e.g., Google instant answer / knowledge panel displays the answer directly).

Row 2: ACTION_OPTIMIZE_SNIPPET | Position 4 with large search volume. What would make it wrong: Search intent is navigational for a direct competitor brand name.

Row 3: ACTION_EXPAND_CONTENT | High potential ranking just off page 1. What would make it wrong: Page is intentionally concise (e.g., a login or utility calculator tool page).

Row 4: ACTION_OPTIMIZE_SNIPPET | Page 1 striking distance. What would make it wrong: Low CTR is caused by aggressive SERP ads pushing organic results below the fold.

Row 5: ACTION_OPTIMIZE_SNIPPET | High impressions at position 5. What would make it wrong: Ranking is volatile and fluctuating daily due to algorithmic testing.

Row 6: ACTION_EXPAND_CONTENT | Low word count on competitive term. What would make it wrong: Competitor winning pages are video-heavy formats, not text.

Row 7: ACTION_OPTIMIZE_SNIPPET | Low CTR relative to page 1 position. What would make it wrong: Seasonal keyword traffic spiking temporarily for an unrepeatable event.

Row 8: ACTION_OPTIMIZE_SNIPPET | High impressions with generic snippet. What would make it wrong: Canonical URL duplicate exists elsewhere on the site.

Row 9: ACTION_EXPAND_CONTENT | Off-page 1 with steady impressions. What would make it wrong: Backlink deficit vs top 3 competitors is the real bottleneck, not content length.

Row 10: ACTION_OPTIMIZE_SNIPPET | Underperforming CTR at position 6. What would make it wrong: Search query intent is informational while the landing page is purely transactional.

In [7]:
import os
import pandas as pd
import numpy as np

# Ensure output directory exists
os.makedirs("../outputs", exist_ok=True)

## 4. Weak picks + leakage check

The heuristic relies purely on raw historical numbers ($CTR$, impressions, position) without understanding semantic intent.It falsely prioritizes high-impression branded or informational instant-answer queries where CTR is inherently low across all competitors.

In [8]:
import os
import pandas as pd
import numpy as np

# Ensure output directory exists
os.makedirs("../outputs", exist_ok=True)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.